# Lab 4.3 — G-Retriever: GNN-Scored Retrieval on HotpotQA
**Module IV · LLMs + GNNs Together (Capstone)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_solutions/blob/main/module-4-hybrid/lab4_3_g_retriever.ipynb)

---

## What you will do
1. Build a **per-question paragraph graph** — the same 10-node graph from Lab 4.2, but now used as training data for a GNN.
2. Train a **GCN for supporting-fact classification**: given a question and its 10 candidate paragraphs, classify each paragraph as *supporting* (class 1) or *distractor* (class 0). This is node classification on a graph — Module III applied to retrieval.
3. Use the trained GCN as a **retrieval re-ranker**: score the 10 paragraphs by their GCN probability and feed the top-2 to the LLM.
4. Run the **grand comparison** — Naive LLM → RAG → GraphRAG → G-Retriever — on the same HotpotQA questions.
5. `[Extension]` Analyse performance by question type and difficulty.

## The connection to He et al. (2024)
The original G-Retriever paper (He et al., NeurIPS 2024) applies this idea to **WebQSP** and **ExplaGraphs** — knowledge-graph benchmarks where each question has a pre-built Freebase or ConceptNet subgraph. The core contribution is the same: a GNN encodes a retrieved subgraph, and those structure-aware representations guide generation.

In this lab we implement the retrieval side of G-Retriever using HotpotQA, where the "knowledge graph" is the per-question paragraph similarity graph. This is conceptually identical and requires no external knowledge base.

## Prerequisites
Lab 4.2 completed (HotpotQA loaded and paragraph graphs built).

**Estimated time:** 65–80 min

---
## 0 · Setup and data reload

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Cloning the course solutions repository and installing dependencies...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_solutions.git"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         "llm-gnns-course_solutions/environment/requirements.txt"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "torch-geometric"], check=True)
    sys.path.insert(0, "llm-gnns-course_solutions")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import re
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from utils import SimpleLLM

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
st_model  = SentenceTransformer("all-MiniLM-L6-v2")
llm       = SimpleLLM()

print(f"Device: {DEVICE}")

In [ ]:
# Reload HotpotQA and recompute embeddings (same as Lab 4.2)
print("Loading HotpotQA ...")
hotpot_val = load_dataset("hotpot_qa", "distractor",
                          split="validation", trust_remote_code=True)
hotpot_train = load_dataset("hotpot_qa", "distractor",
                            split="train", trust_remote_code=True)

def parse_example(ex):
    titles    = ex["context"]["title"]
    sentences = ex["context"]["sentences"]
    sup_titles = set(ex["supporting_facts"]["title"])
    return {
        "id":         ex["id"],
        "question":   ex["question"],
        "answer":     ex["answer"],
        "type":       ex["type"],
        "level":      ex["level"],
        "paragraphs": [
            {"title": t, "text": " ".join(s),
             "is_supporting": t in sup_titles}
            for t, s in zip(titles, sentences)
        ],
    }


def embed_examples(examples, batch_size=64):
    """Embed questions and all paragraphs; store in-place."""
    q_texts = [ex["question"] for ex in examples]
    p_texts = [p["text"] for ex in examples for p in ex["paragraphs"]]

    q_embs = st_model.encode(q_texts, batch_size=batch_size,
                             show_progress_bar=True, convert_to_numpy=True
                             ).astype(np.float32)
    p_embs = st_model.encode(p_texts, batch_size=batch_size,
                             show_progress_bar=True, convert_to_numpy=True
                             ).astype(np.float32)

    q_embs /= np.linalg.norm(q_embs, axis=1, keepdims=True)
    p_embs /= np.linalg.norm(p_embs, axis=1, keepdims=True)

    for i, ex in enumerate(examples):
        ex["question_embedding"] = q_embs[i]
        for j, p in enumerate(ex["paragraphs"]):
            p["embedding"] = p_embs[i * 10 + j]


# 1 000 training + 300 evaluation examples
N_TRAIN, N_EVAL = 1000, 300
print(f"Parsing {N_TRAIN} train + {N_EVAL} eval examples ...")
train_examples = [parse_example(hotpot_train[i]) for i in range(N_TRAIN)]
eval_examples  = [parse_example(hotpot_val[i])   for i in range(N_EVAL)]

print("Embedding training examples ...")
embed_examples(train_examples)
print("Embedding evaluation examples ...")
embed_examples(eval_examples)

bridge_eval  = [ex for ex in eval_examples if ex["type"] == "bridge"]
compare_eval = [ex for ex in eval_examples if ex["type"] == "comparison"]
print(f"\nTrain: {N_TRAIN} | Eval: {N_EVAL} "
      f"(bridge={len(bridge_eval)}, comparison={len(compare_eval)})")

---
## 1 · Per-question paragraph graphs as a GNN training dataset

Each HotpotQA example becomes a **small graph** for node classification:

| Element | Description |
|---|---|
| Nodes | 10 candidate paragraphs |
| Node features | Sentence-transformer embedding of the paragraph (384-dim) |
| Edges | Pairs with cosine similarity > threshold |
| Node labels | 1 = supporting fact, 0 = distractor |

This is **identical to node classification on Cora** (Module III), just with much smaller graphs (10 nodes instead of 2,708) and a binary task. We train one shared GCN across all question-graphs — the GCN learns structural patterns that help distinguish supporting facts from distractors regardless of the specific question topic.

**Why should the graph structure help?** Supporting paragraphs for bridge questions share an intermediate entity (the bridge entity). They are more likely to be semantically connected in the paragraph graph than two random distractors. A GCN that sees the graph can exploit this pattern; a plain MLP (scoring each paragraph in isolation) cannot.

### Exercise 4.3.1 `[Core]` — Convert HotpotQA examples to PyG Data objects

Implement `example_to_pyg(ex, threshold=0.5)` that converts one HotpotQA example into a PyG `Data` object with:
- `x`: shape `(10, 384)` — paragraph embeddings as node features.
- `edge_index`: edges for paragraph pairs with cosine similarity > `threshold`.
- `y`: shape `(10,)` — binary labels (1 = supporting, 0 = distractor).

Then build `train_dataset` (1,000 graphs) and `eval_dataset` (300 graphs). Report: average number of edges per graph and average number of positive labels.

In [ ]:
# YOUR CODE HERE
# Hint: Implement example_to_pyg(ex, threshold=0.5) that converts one HotpotQA example into a PyG Data object with:

def example_to_pyg(ex: dict, threshold: float = 0.5) -> Data:
    pass  # replace this

---
## 2 · Training the GCN for supporting-fact classification

We train a 2-layer GCN using the same architecture as Module III, but now the task is **binary node classification** (supporting vs. distractor) across 1,000 question-graphs in mini-batches.

PyG's `DataLoader` handles mini-batching automatically: it stacks multiple disconnected question-graphs into one large batch-graph, so all 32 question-graphs in a batch are processed in one forward pass. The node labels and masks are preserved per sub-graph.

**Class imbalance:** Every graph has exactly 2 positives and 8 negatives (80% negative). We compensate with a `pos_weight` in `BCEWithLogitsLoss`.

### Exercise 4.3.2 `[Core]` — Build and train the supporting-fact GCN

1. Implement `SupportingFactGCN` — a 2-layer GCN producing a scalar logit per node.
2. Train it for 100 epochs on `train_dataset` using mini-batches of size 32. Use `pos_weight=4.0` to handle the 4:1 class imbalance.
3. Print training loss every 20 epochs.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Implement SupportingFactGCN — a 2-layer GCN producing a scalar logit per node.

class SupportingFactGCN(nn.Module):
    pass  # replace this

### Exercise 4.3.3 `[Core]` — Evaluate node classification on the training data

Evaluate the trained GCN on `train_dataset` (in-sample) and `eval_dataset` (out-of-sample):
1. For each graph, score all 10 nodes with the GCN.
2. Predict class 1 (supporting) for the top-2 scoring nodes.
3. Compute **graph-level accuracy**: fraction of graphs where both predicted positives are truly supporting.

Compare in-sample vs. out-of-sample accuracy. Does the GCN generalise across questions?

In [ ]:
# YOUR CODE HERE
# Hint: Evaluate the trained GCN on train_dataset (in-sample) and eval_dataset (out-of-sample):
raise NotImplementedError("Complete this exercise")

> **Interpretation:** A GCN accuracy well above the random baseline (1/45 ≈ 2.2%) confirms the model has learned something meaningful about what makes a paragraph a supporting fact. The gap between in-sample and out-of-sample accuracy reflects the difficulty of generalising across different Wikipedia topics. The GCN sees each question's graph in isolation at inference time, so it must learn graph-structural patterns that transfer across topics.

---
## 3 · G-Retriever: GCN-scored retrieval

We now use the trained GCN as a **learned retrieval re-ranker**: instead of ranking paragraphs by cosine similarity with the question, we rank them by the GCN's predicted probability of being a supporting fact.

This is the core retrieval contribution of G-Retriever: the GNN uses **graph structure** (which paragraphs are connected to which) alongside **node features** (paragraph text) to score each candidate. Plain cosine similarity ignores the relationships between paragraphs.

In [ ]:
def full_recall_at_k(examples_list, retrieve_fn, k=2):
    correct = 0
    for ex in examples_list:
        retrieved  = retrieve_fn(ex)[:k]
        ret_titles = {p["title"] for p in retrieved}
        sup_titles = {p["title"] for p in ex["paragraphs"] if p["is_supporting"]}
        if sup_titles.issubset(ret_titles):
            correct += 1
    return correct / len(examples_list)


def rag_retrieve(ex):
    q_emb  = ex["question_embedding"]
    p_embs = np.stack([p["embedding"] for p in ex["paragraphs"]])
    sims   = p_embs @ q_emb
    order  = np.argsort(-sims)
    return [ex["paragraphs"][i] for i in order]


def graphrag_retrieve(ex, k_seed=1, hops=1, threshold=THRESHOLD):
    paras  = ex["paragraphs"]
    q_emb  = ex["question_embedding"]
    p_embs = np.stack([p["embedding"] for p in paras])
    sims   = p_embs @ q_emb

    seed = set(np.argsort(-sims)[:k_seed].tolist())
    pyg  = example_to_pyg(ex, threshold)
    G    = nx.Graph()
    G.add_nodes_from(range(10))
    ei   = pyg.edge_index.numpy()
    for s, d in zip(ei[0], ei[1]):
        if s < d:
            G.add_edge(int(s), int(d))

    expanded, frontier = set(seed), set(seed)
    for _ in range(hops):
        new_nodes = set()
        for n in frontier:
            new_nodes.update(G.neighbors(n))
        new_nodes -= expanded
        expanded.update(new_nodes)
        frontier = new_nodes

    ranked = sorted(expanded, key=lambda i: -sims[i])
    return [paras[i] for i in ranked]


@torch.no_grad()
def g_retriever_retrieve(ex):
    gcn.eval()
    data   = example_to_pyg(ex, THRESHOLD).to(DEVICE)
    logits = gcn(data.x, data.edge_index).cpu()  # (10,)
    probs  = torch.sigmoid(logits).numpy()
    order  = np.argsort(-probs)
    return [ex["paragraphs"][i] for i in order]


print("All retrieval functions defined.")

### Exercise 4.3.4 `[Core]` — Full Retrieval Recall: all three systems

Compute Full Retrieval Recall@2 for RAG, GraphRAG, and G-Retriever on the 300 evaluation examples (and separately by question type). Print a comparison table.

In [ ]:
# YOUR CODE HERE
# Hint: Compute Full Retrieval Recall@2 for RAG, GraphRAG, and G-Retriever on the 300 evaluation examples (and separately by ...
raise NotImplementedError("Complete this exercise")

In [ ]:
# Grouped bar chart
subset_names = [s for s, _ in subsets]
x = np.arange(len(subset_names))
width = 0.25
colors = ["#3498db", "#27ae60", "#e67e22"]

fig, ax = plt.subplots(figsize=(9, 5))
for k, (sys_name, _) in enumerate(systems):
    vals = [recalls[sys_name][s] for s in subset_names]
    ax.bar(x + (k - 1) * width, vals, width,
           label=sys_name, color=colors[k], edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(subset_names)
ax.set_ylabel("Full Retrieval Recall@2")
ax.set_title("Retrieval Recall@2 — HotpotQA distractor (n=300)")
ax.set_ylim(0, 1.05)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

> **What to expect:** G-Retriever should outperform plain RAG on bridge questions, and should be competitive with (or better than) GraphRAG. On comparison questions, all three systems should be close — the graph structure provides less additional signal when the two supporting paragraphs are independently relevant to the question.
>
> The GCN gains come from two sources: (1) message passing propagates information from high-confidence paragraphs to their neighbours, boosting the score of the second supporting paragraph that shares an entity with the first; (2) the GCN was *trained with supervision* — it has seen 1,000 examples of what supporting paragraphs look like in terms of graph position.

---
## 4 · End-to-end QA: Naive LLM → RAG → GraphRAG → G-Retriever

### Exercise 4.3.5 `[Core]` — Grand comparison on 30 bridge questions

Run all four systems on 30 bridge questions from the eval set. Measure **answer accuracy** = fraction of questions where the ground-truth answer appears in the generated text.

Systems:
- **Naive LLM**: no retrieval — the LLM answers from parametric memory alone.
- **RAG**: cosine-similarity top-2 → LLM.
- **GraphRAG**: seed + 1-hop expansion top-2 → LLM.
- **G-Retriever**: GCN-scored top-2 → LLM.

In [ ]:
# YOUR CODE HERE
# Hint: Run all four systems on 30 bridge questions from the eval set. Measure answer accuracy = fraction of questions where ...

def normalize(s):
    pass  # replace this

In [ ]:
# Summary chart
sys_names = list(qa_results.keys())
accs = [qa_results[n] for n in sys_names]
bar_colors = ["#95a5a6", "#3498db", "#27ae60", "#e67e22"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(sys_names, accs, color=bar_colors, edgecolor="white", width=0.55)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.01,
            f"{acc:.0%}", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("Answer accuracy")
ax.set_ylim(0, 1.05)
ax.set_title(f"End-to-end QA accuracy — {N_QA} HotpotQA bridge questions")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## 5 · `[Extension]` Performance by question difficulty

### Exercise 4.3.6 `[Extension]` — Stratified retrieval recall by difficulty

HotpotQA labels each question as easy, medium, or hard. Compute Full Retrieval Recall@2 for all three systems separately on easy, medium, and hard **bridge** questions. Which system benefits most on hard questions? Does graph structure matter more as questions get harder?

In [ ]:
# YOUR CODE HERE
# Hint: HotpotQA labels each question as easy, medium, or hard. Compute Full Retrieval Recall@2 for all three systems separat...
raise NotImplementedError("Complete this exercise")

---
## 6 · Connection to the G-Retriever paper

He et al. (2024) apply the same GNN-for-retrieval idea at a larger scale:

| Aspect | Our implementation | He et al. (2024) |
|---|---|---|
| Dataset | HotpotQA (Wikipedia para. graphs) | WebQSP + ExplaGraphs (Freebase / ConceptNet KGs) |
| Graph | Per-question similarity graph (10 nodes) | Per-question KG subgraph (variable size) |
| GNN | 2-layer GCN, node classification | GNN encoder + mean pooling |
| LLM interface | Top-k paragraphs as text context | Soft token prepended to LLM input |
| Training | Supervised retrieval (supporting fact labels) | End-to-end with generation loss |

The core principle is shared: **encode a retrieved subgraph with a GNN to produce structure-aware representations, then use those representations to guide the LLM**. Our implementation captures the retrieval side; the paper also integrates GNN embeddings directly into the LLM's attention (soft prompting), which requires fine-tuning the LLM — beyond a laptop-friendly lab.

WebQSP (used in the paper) is accessible via the PyG dataset class `torch_geometric.datasets.WebQSP` for those who want to reproduce the paper's exact experimental setup.

---
## Course Summary

### The progression we built

| Lab | System | Dataset | Key lesson |
|---|---|---|---|
| 2.1–2.2 | Prompting, chatbot | TechRetail (8 docs) | LLMs hallucinate without grounding |
| 2.3 | Basic RAG | TechRetail | Retrieval fixes factual errors |
| 3.1–3.3 | GCN, GAT | Cora (2,708 nodes) | Graph structure adds 20 pp over features alone |
| 4.1 | LLM features + GCN | Synthetic (48 papers) | LLM embeddings + graph = best of both |
| 4.2 | GraphRAG | **HotpotQA (300 Qs)** | Graph walk improves multi-hop retrieval |
| 4.3 | G-Retriever | **HotpotQA (1k train / 300 eval)** | Supervised GCN learns what dense retrieval misses |

### Retrieval system comparison on HotpotQA bridge questions

| System | Full Recall@2 | Answer acc. | Retrieval mechanism |
|---|---|---|---|
| Naive LLM | — | ~20–30% | None (parametric memory) |
| RAG | ~45–55% | ~40–50% | Cosine similarity |
| GraphRAG | ~55–65% | ~50–60% | Cosine + 1-hop graph walk |
| G-Retriever | ~60–70% | ~55–65% | Supervised GCN scoring |
| SotA (MDR, Baleen) | >90% | >80% | Multi-step chain retrieval |

*Approximate ranges over random 300-example subsets.*

### What's left to close the gap to state-of-the-art?
- **Multi-step retrieval** (MDR, Baleen): retrieve iteratively, conditioning each step on what was found so far.
- **End-to-end training** of GNN + LLM jointly (G-Retriever's soft prompting).
- **Graph Transformers** (GPS, Graphormer): global attention over the paragraph graph instead of local GCN aggregation.
- **Relational Deep Learning** (Fey et al., ICLR 2024): scale to full enterprise relational databases with heterogeneous entity types.